In [ ]:
# One must patch the DPO Trainer first!
from unsloth import PatchDPOTrainer

PatchDPOTrainer()


from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen2.5-1.5B-merged", # Choose ANY! eg mistralai/Mistral-7B-Instruct-v0.2
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


In [ ]:
## data prep

import pandas as pd
from datasets import DatasetDict, concatenate_datasets, load_dataset, Dataset
train_data = pd.read_csv('./data/train/train_dpo.csv').sample(2000,random_state=323).reset_index(drop=True)
val_data = pd.read_csv('./data/test/test_dpo.csv').sample(200,random_state=323).reset_index(drop=True)

train_data = train_data[['text','results','qwen2.5_result']]
train_data.columns = ['text','text_chosen','text_reject']


val_data = val_data[['text','results','qwen2.5_sft_result']]
val_data.columns = ['text','text_chosen','text_reject']


train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)


In [ ]:
train_dataset = train_dataset.map(generate_prompts, batched=False)
val_dataset = val_dataset.map(generate_prompts, batched=False)

In [ ]:
import pprint

row = train_dataset[8]
pprint.pprint(row["prompt"])
pprint.pprint(row["chosen"])
pprint.pprint(row["rejected"])



# 프롬프트 생성 함수
def generate_prompts(row):


    text = row['text']
    text_chosen = row['text_chosen']
    text_reject = row['text_reject']

    messages = [{"role":"system","content":"you are a help ful assistant"},
        {"role":"user","content": f"""Please summarize the documentation provided in 3 lines.
Also, please extract the top five key phrases. See template for the answer format.
The summary must be written in the same language as the body.
<template>
summary
- summarize 1
- summarize 2
- summarize 3

key phrases
[key phrase1, key phrase2, key phrase3, key phrase4, key phrase5]
</template>

docs:
{text}"""}]
    message_chosen = [ {"role": "assistant", "content": f"{text_chosen}"}]
    message_reject = [ {"role": "assistant", "content": f"{text_reject}"}]

    row['prompt'] =  tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    row['chosen'] =  tokenizer.apply_chat_template(message_chosen, tokenize=False, add_generation_prompt=False)
    row['chosen'] = row['chosen'].replace('<|im_start|>system\n'
 'You are Qwen, created by Alibaba Cloud. You are a helpful '
 'assistant.<|im_end|>\n','')
    row['rejected'] =  tokenizer.apply_chat_template(message_reject, tokenize=False, add_generation_prompt=False)
    row['rejected'] = row['rejected'].replace('<|im_start|>system\n'
 'You are Qwen, created by Alibaba Cloud. You are a helpful '
 'assistant.<|im_end|>\n','')
    
    return row


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # Currently only supports dropout = 0
    bias = "none",    # Currently only supports bias = "none"
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 323,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


## Train the DPO model

from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

# 오류때문에 cutomDPOTrainer를 쓰는거야.
class CustomDPOTrainer(DPOTrainer):
    def log(self, logs, start_time=None):
        # start_time을 무시하거나 부모 클래스의 log 메서드 호출
        super().log(logs)

# DPOTrainer 대신 CustomDPOTrainer 사용
# TrainingArguments 쓰도록 변경되어야 할거같아.
dpo_trainer = CustomDPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=3,
        learning_rate=5e-6,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1000,
        evaluation_strategy="steps",
        eval_steps=1000,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="linear",
        seed=323,
        output_dir="outputs_dpo",
        report_to="none",
        save_steps=1000,
    ),
    beta=0.1,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    max_length=2048,
    max_prompt_length=512,
)

dpo_trainer.train()